# 02 — Поиск по Qdrant и проверочные вопросы

Выполняет поиск без reranker по чанкам 350 токенов с перекрытием 50 токенов по 30 заранее заданным вопросам. Для каждого вопроса показывает топ найденных фрагментов, дату, чат, автора, score и ссылки на исходные PDF/изображения. Ниже есть необязательный режим формирования краткого ответа локальной моделью Ollama; он использует только найденные фрагменты.

In [ ]:
# При необходимости в kernel этого ноутбука выполните:
# %pip install "sentence-transformers==5.2.2" "transformers==4.57.4" "qdrant-client>=1.19" pandas tqdm requests

In [ ]:
from pathlib import Path
import requests
from IPython.display import display, Markdown
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data").is_dir()
)
QDRANT_URL = "http://localhost:6333"
COLLECTION = "telegram_keep_user2_350tok_50overlap"
MODEL_NAME = "deepvk/USER2-base"
TOP_K = 5
GENERATE_ANSWERS = False  # True: дополнительно попросить Ollama сформулировать ответ
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "qwen3.5:4b"
client = QdrantClient(url=QDRANT_URL, timeout=30)
model = SentenceTransformer(MODEL_NAME)
assert any(c.name == COLLECTION for c in client.get_collections().collections), (
    f"Нет коллекции {COLLECTION}; сначала выполните 01_qdrant_ingest_user2.ipynb"
)

## Контрольные вопросы (30)

In [ ]:
QUESTIONS = [
    "Что известно о контрольной работе по теории вероятностей «Хайзенберг» и что лежит в PDF?",
    "Какие темы по C++ обсуждаются в чате и какие материалы приложены?",
    "Чем отличаются vector, deque и list в материалах или обсуждениях?",
    "Что в чате говорится о конструкторах классов в C++?",
    "Какие материалы объясняют работу с памятью в C++?",
    "Что говорится о дружественных классах и виртуальных функциях?",
    "Какие документы или сообщения посвящены дифференциальным уравнениям?",
    "Что известно о семестровой работе по дифференциальным уравнениям?",
    "Какие темы перечислены в раздаточном материале по дифференциальным уравнениям?",
    "Как менялось расписание занятий и какие пары переносили?",
    "Как студентов распределяли по группам и подгруппам?",
    "Что делать, если возникли проблемы с оформлением или системой 1С?",
    "Что нужно сдать или закрыть для допуска к экзамену по 1С ERP?",
    "Как устроен экзамен по 1С ERP и сколько вопросов в билете?",
    "Какие сведения об оплате обучения или реквизитах КубГУ публиковали?",
    "Как оплатить обучение и где найти QR-код?",
    "Какие методички, планы или учебные документы выкладывали в чате?",
    "Кто упоминался как староста и что говорили о распределении по группам?",
    "Где найти файл по лабораторной работе 2 физического практикума?",
    "Как распределяют варианты контрольных и как определить свой номер?",
    "Когда ожидали расписание или доступ к учебному диску?",
    "Какие материалы по алгоритмам и структурам данных доступны?",
    "Какие материалы для экзамена по теории вероятностей есть в чате?",
    "Есть ли список вопросов к экзамену и когда его обещали выложить?",
    "Какие домашние работы по молекулярной физике нужно выполнить и к какому сроку?",
    "Что сказано про пересдачи и долги в конце учебного года?",
    "Какой номер деканата заочного отделения искали в переписке?",
    "Какие даты и условия сдачи контрольных по физике упоминались?",
    "Какие прикреплённые PDF содержат учебные задания или контрольные?",
    "Что сообщали о переходе студентов на второй курс?",
]

In [ ]:
def search(question, limit=TOP_K):
    vector = model.encode(
        question, prompt_name="search_query", convert_to_numpy=True
    ).tolist()
    hits = client.query_points(
        collection_name=COLLECTION, query=vector, limit=limit, with_payload=True
    ).points
    return hits


def link_for(path):
    if not path:
        return ""
    return f"[{Path(path).name}](<{Path(path).resolve()}>)"


def answer_with_ollama(question, hits):
    context = "\n\n".join(
        f"[{i + 1}] {h.payload.get('date')} | {h.payload.get('chat_name')} | {h.payload.get('sender')}\n{h.payload.get('text')}\nФайлы: {', '.join(h.payload.get('file_paths', []))}"
        for i, h in enumerate(hits)
    )
    prompt = f"Ответь по-русски только по контексту. Если данных недостаточно, скажи об этом. Указывай [номер] фрагмента.\n\nВопрос: {question}\n\nКонтекст:\n{context}"
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["message"]["content"]


results = []
for qi, question in enumerate(QUESTIONS, 1):
    hits = search(question)
    display(Markdown(f"### {qi}. {question}"))
    if not hits:
        display(Markdown("_Ничего не найдено._"))
        continue
    for rank, hit in enumerate(hits, 1):
        p = hit.payload
        links = [link_for(x) for x in p.get("file_paths", [])]
        links = [x for x in links if x]
        display(
            Markdown(
                f"**{rank}. score={hit.score:.3f}** · {p.get('date')} · {p.get('chat_name')} · {p.get('sender')}\n\n{p.get('text', '')}\n\n"
                + ("Файлы: " + ", ".join(links) if links else "Файлы: нет")
            )
        )
        results.append(
            {
                "question": question,
                "rank": rank,
                "score": hit.score,
                "date": p.get("date"),
                "chat_name": p.get("chat_name"),
                "sender": p.get("sender"),
                "text": p.get("text"),
                "record_id": p.get("record_id"),
                "file_paths": p.get("file_paths"),
                "pdf_paths": p.get("pdf_paths"),
            }
        )
    if GENERATE_ANSWERS:
        display(
            Markdown(
                "**Ответ по найденным фрагментам:**\n\n"
                + answer_with_ollama(question, hits)
            )
        )

results_df = pd.DataFrame(results)
print(f"Запросов: {len(QUESTIONS)}, найденных фрагментов: {len(results_df)}")

## Сводная таблица для проверки качества поиска и ответов

Сортируйте или фильтруйте `results_df`. Ответы формируются локальной моделью Ollama `qwen3.5:4b` по пяти найденным фрагментам без reranker. Чтобы отключить генерацию и оценивать только retrieval, задайте `GENERATE_ANSWERS=False`.

In [ ]:
display(
    results_df[["question", "rank", "score", "date", "chat_name", "text"]].head(50)
    if len(results_df)
    else results_df
)
# При желании сохранить результаты локально:
# results_df.to_csv(ROOT / 'data/processed/qdrant_search_eval_user2.csv', index=False, encoding='utf-8-sig')